# Small AlexNet Implementation
Adapted from the original AlexNet model by Krizhevsky, et al. A more modern approach of this model is to use Batch Normalisation between layers instead of the original Local Response Normalisation.For training and tuning, this model was used in place of the full scale model. ONce desired performance was reached, full model implementation took place.

*Dataset:*
- 20 class subset of iNaturalist-2021 (mini)
- 30 images per class (20:10 train-test split)
- Data samples processed in src/data_processing/sampling_dataset.ipynb


## Setup

In this stage, we import the necessary libraries

In [4]:
import csv
import time
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import optuna

## Data pipeline

The input layer of AlexNet requirs images to be resized to a 224x224x3 tensor. The following transforms are performed:
1) training set:
   1) random resized crop (plus a smale horizontal scale adjustment)
   2) random horizontal filp
   3) to tensor
   4) normalise
2) validation/test set:
   1) resize
   2) centre crop
   3) to tensor
   4) normalise
The normalisation transform will involve the mean and standard deviation commonly used in ImageNet normalisation techniques

In [5]:
# Image transformation environment variables
# IMG_SIZE = 128
IMG_SIZE = 227
# IMG_SIZE = 256
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# dataset paths set up
DATA_ROOT = Path("C:/dataset/sampled_20")
# DATA_ROOT = Path("C:/dataset/sampled_500")
TRAIN_PATH = DATA_ROOT / "train_mini"
VALIDATION_PATH = DATA_ROOT / "validation"
TEST_PATH = DATA_ROOT / "val"

# establish transforms for all sub-datasets
train_set_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

val_test_set_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# load datasets uses these transforms and dataset paths
def load_datasets():
    train_ds = datasets.ImageFolder(TRAIN_PATH, transform=train_set_transform)
    validation_ds = datasets.ImageFolder(VALIDATION_PATH, transform=val_test_set_transform)
    test_ds = datasets.ImageFolder(TEST_PATH, transform=val_test_set_transform)
    
    return train_ds, validation_ds, test_ds


In [6]:
# load datasets 
train_ds, val_ds, test_ds = load_datasets()

print(f"classes found: {len(train_ds.classes)}")
print(f"train images:  {len(train_ds)}")
print(f"val images:    {len(val_ds)}")
print(f"test images:   {len(test_ds)}")

image, label = train_ds[0]
print(f"sample tensor shape: {image.shape}")
print(f"sample label index:  {label} -> class '{train_ds.classes[label]}'")

FileNotFoundError: [Errno 2] No such file or directory: 'C:/dataset/sampled_20/train_mini'

## Model Architecture

As mentioned previously, the input layer is a 224x224x3 tensor. Before convolution layers 2, 3 and 5, Max Pooling also takes place. After all the convolution layers, flattening takes place before entering the final fully connected layers.
Below are the remaining layers of the proposed AlexNet model:
|Layer|Implementation|Kernel Size|Stride|Padding|Output Dimension|
|:---|:---|:---|:---|:---|:----|
|Convolution 1|`Conv2d`|11x11|4|0|54x54x96|
|Max Pooling|`MaxPool2d`|3x3|2|0|26x26x96|
|Convolution 2|`Conv2d`|5x5|1|2|26x26x256|
|Max Pooling|`MaxPool2d`|3x3|2|0|12x12x256|
|Convolution 3|`Conv2d`|3x3|1|1|12x12x384|
|Convolution 4|`Conv2d`|3x3|1|1|12x12x384|
|Convolution 5|`Conv2d`|3x3|1|1|12x12x256|
|Max Pooling|`MaxPool2d`|3x3|2|0|5x5x256|
|Flatten|||||6400|
|Fully Connected Layer 1|`Linear`||||1024 (usually 4096)|
|Fully Connected Layer 2|`Linear`||||1024 (usually 4096)|
|Fully Connected Layer 3|`Linear`||||500|

A large risk attached to this model is the large size of the Fully Connected Layers. With millions of parameters at play, the model can easily overfit on a small dataset. To prevent this, dropout is used before the first two fully connected layers. This allows only a select number of nodes in each layer to be used based on a given probability.
This helps significantly reduce the number of parameters used.
For the actual implementation in torch, the convolution layers are described as features, and the fully connected layers are described as classifier.


In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=500, input_size=224, fc_width=4096, dropout=0.5):
        super().__init__()
        
        self.features = nn.Sequential(
            # Conv1: 224x224x3 -> 54x54x96 -> pool -> 26x26x96
            nn.Conv2d(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0),
            nn.BatchNorm2d(num_features=96),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Conv2: 26x26x96 -> 26x26x256 -> pool -> 12x12x256
            nn.Conv2d(in_channels=96, out_channels=256, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(num_features=256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Conv3: 12x12x256 -> 12x12x384
            nn.Conv2d(in_channels=256, out_channels=384, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=384),
            nn.ReLU(inplace=True),

            # Conv4: 12x12x384 -> 12x12x384
            nn.Conv2d(in_channels=384, out_channels=384, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=384),
            nn.ReLU(inplace=True),

            # Conv5: 12x12x384 -> 12x12x256 -> pool -> 5x5x256
            nn.Conv2d(in_channels=384, out_channels=256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, 3, input_size, input_size)
            flat_size = self.features(dummy).flatten(start_dim=1).size(1)
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(flat_size, fc_width),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(fc_width, fc_width),
            nn.ReLU(inplace=True),
            nn.Linear(fc_width, num_classes),
        )
        
    def forward(self, x):
        x = self.features(x)
        # keep batch dim, flatten the rest
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x
        

## Training setup

Design decisions, and why they differ from a typical transfer-learning setup:

- **SGD with momentum + Nesterov**, not Adam. Scratch CNNs on small datasets tend to generalise
  better with SGD; Adam often converges faster but overfits harder in this regime.
- **Label smoothing (0.1)** on the loss — softens the training targets slightly, a cheap
  regulariser that helps given how overparameterised this model is relative to 20,000 images.
- **Cosine annealing** learning rate schedule — decays smoothly to zero rather than in steps.
- A **higher starting learning rate** (0.1) than a pretrained model would use (typically ~1e-4),
  since random-init weights need larger updates early on to move away from their starting point.

In [ ]:
# setting the torch seed and device for training
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_WORKERS = 2
SEED = 42

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

model = AlexNet(num_classes=len(train_ds.classes), input_size=IMG_SIZE, fc_width=2048, dropout=0.5).to(device=device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimiser = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=0.005, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)

## Training Loop
Training loop implementation handles which set is being used. Training set enables gradient updates, test set provides no weight updates. 

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)

    return total_loss / total, correct / total

## Running Training and Logging Results

Logs each result to a csv logging file which can be used officially in the report. 

In [ ]:
out_dir = Path("runs/scratch")
out_dir.mkdir(parents=True, exist_ok=True)

log_path = out_dir / "small_log.csv"
with open(log_path, "w", newline="") as f:
    csv.writer(f).writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "lr"])

best_val_acc = 0.0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimiser, device, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimiser, device, train=False)
    scheduler.step()

    current_lr = optimiser.param_groups[0]["lr"]
    print(f"epoch {epoch}/{EPOCHS} "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} lr={current_lr:.2e}")

    with open(log_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch, train_loss, train_acc, val_loss, val_acc, current_lr])

    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "classes": train_ds.classes,
        "mode": "scratch",
        "val_acc": val_acc,
    }
    torch.save(checkpoint, out_dir / "small_last.pt")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(checkpoint, out_dir / "small_best.pt")

total_time = time.time() - start_time
print(f"\nDone. Best val acc: {best_val_acc:.4f}. Total training time: {total_time/60:.1f} min")

## Hyperparameter search ~ Determine best hyperparameters (Optuna)

Manual tuning so far has shown dropout strength matters more than `fc_width` for controlling
the overfitting gap, and both interact with `weight_decay`. Rather than continuing to hand-pick
values, this runs an automated search over the three at once, using this notebook's existing
setup.

Each trial trains a fresh model for a reduced number of epochs and reports back val accuracy.
Optuna's pruner can stop clearly-bad trials early rather than running them to completion, saving
time on a CPU-bound search.

**This does not replace the training loop above** -- it searches for the best config here, then
that config gets plugged into the real training run (either the cell above with updated
hyperparameters, or the full-scale run) afterwards.

In [ ]:
SEARCH_EPOCHS=50
N_TRIALS=50

def objective(trial):
    dropout = trial.suggest_float("dropout", 0.4, 0.9)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 5e-3, log=True)
    fc_width = trial.suggest_categorical("fc_width", [1024, 2048, 4096])
    momentum = trial.suggest_float("momentum", 0.5, 0.9)
    
    set_seed(SEED)
    trial_model = AlexNet(
        num_classes=len(train_ds.classes),
        input_size=IMG_SIZE,
        fc_width=fc_width,
        dropout=dropout
    ).to(device)
    
    trial_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    trial_optimiser = torch.optim.SGD(trial_model.parameters(), lr=LEARNING_RATE, momentum=momentum, weight_decay=weight_decay, nesterov=True)
    trial_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(trial_optimiser, T_max=SEARCH_EPOCHS)
    
    val_acc = 0.0
    for epoch in range(SEARCH_EPOCHS):
        run_epoch(trial_model, train_loader, trial_criterion, trial_optimiser, device, train=True)
        _, val_acc = run_epoch(trial_model, val_loader, trial_criterion, trial_optimiser, device, train=False)
        trial_scheduler.step()

        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return val_acc


study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=N_TRIALS)

print(f"\nBest val acc: {study.best_value:.4f}")
print(f"Best params:  {study.best_params}")

trials_df = study.trials_dataframe().sort_values("value", ascending=False)
trials_df[["number", "value", "params_dropout", "params_momentum", "params_weight_decay", "params_fc_width", "state"]]